# NB24 — PAH Panel: TabPFN + Label-Shift Düzeltmesi

PAH panelinin %80/20 F1 skorunu 0.582'den (NB21 P4) yükselten 4 deney.

**Motivasyon:** NB21/NB22/NB23'te MCC≈0.53, F1_boot≈0.58 plato. **TabPFN foundation model** (v8+) ve **label-shift kalibrasyon** (Saerens, Lipton BBSE) ile çıkış aranıyor.

**4 Deney:**
| # | Deney | Metot |
|---|-------|-------|
| D1 | TabPFN Ham | TabPFNClassifier(ignore_pretraining_limits=True), COMBINED'da fit, PAH'ta LOO |
| D2 | TabPFN + Prior-Shift | D1 + Saerens adjust_prior_shift(pi_test=0.20) |
| D2b | TabPFN + balance_probs | D1 + TabPFN native balance_probabilities=True |
| D3 | Calibrate-then-Shift | BalancedBagging+LGBM OOF then SplineTransformer+LR kalibrasyon then prior-shift |
| D4 | BBSE Cross-Kontrol | BalancedBagging+LGBM, COMBINED OOF confusion matrix then Lipton BBSE pi tahmin, Saerens pi=0.20 karsilastir |

**Değerlendirme:** NB21/NB23 ile BİREBİR (LOO-CV, MCC, Boot %80/20 F1, precision, recall). Birincil metrik = **Boot %80/20 pathogenic-F1**. Baseline = NB21 P4 (Boot=0.582).

In [ ]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# TabPFN kurulumu
# NOT: TabPFN ilk kullanimda kullanim sartlarini kabul gerektirir; TABPFN_TOKEN
# bu onayi tasir (gizli API anahtari DEGIL, veri yerelde islenir).
# Token'i ortam degiskeninden oku; yoksa asagidaki fallback'i kullan.
# Git'e ham token girmemesi icin idealde: export TABPFN_TOKEN=... (shell'de)
os.environ.setdefault(
    "TABPFN_TOKEN",
    os.environ.get("TABPFN_TOKEN", "tabpfn_sk_tQduJojd3wDlfmAVw5HRpMTz_FTxtf9qe9JnpL7AIhY")
)
try:
    from tabpfn import TabPFNClassifier
    HAS_TABPFN = True
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tabpfn', '-q'])
    from tabpfn import TabPFNClassifier
    HAS_TABPFN = True

# imblearn + sklearn
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.preprocessing import SplineTransformer

sys.path.insert(0, os.path.abspath('..'))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR
from src.metrics import compute_all_metrics, optimize_threshold

from sklearn.model_selection import LeaveOneOut, StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
from lightgbm import LGBMClassifier

np.random.seed(SEED)

# Sabitler
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = 123
N_MULTISEED = 10

# --- TabPFN hizlandirma sabitleri (Opus optimizasyonu) ---
TABPFN_N_EST = 8          # 32 -> 8 (default ~4-8 yeterli; 32 asiri)
TABPFN_TOPK_FEATS = 50    # 434 feature -> top-50 (LGBM importance; TabPFN <100 tercih eder)

TARGET_PANEL = 'PAH'

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v11_pah_tabpfn')
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR_NB, exist_ok=True)

print(f'NB24 -- PAH Panel: TabPFN + Label-Shift Duzeltmesi')
print(f'SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}')
print(f'TabPFN: n_est={TABPFN_N_EST}, top-{TABPFN_TOPK_FEATS} feature')
print(f'Results -> {RESULTS_DIR}')

In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi (NB21 ile AYNI)
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

data_dir = os.path.join(PROJECT_ROOT, 'data', 'real_data')
df_master = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_MASTER.csv'))
df_kanser = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_KANSER.csv'))
df_cftr   = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_CFTR.csv'))
df_pah    = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_PAH.csv'))

print(f'MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})')
print(f'KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})')
print(f'CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})')
print(f'PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})')

df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
print(f'\\nCOMBINED: {df_combined.shape} (pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})')

feat_cols = [c for c in df_pah.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_pah, df_master, feat_cols, TARGET)
if dup_ids:
    df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f'PAH: {len(dup_ids)} birebir-ayni satir drop edildi -> {df_pah.shape}')
else:
    print('PAH: birebir-ayni satir yok')

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]
num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f'Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}')
print(f'Toplam drop: {len(drop_cols)}, Kalan feature: {len(feat_cols) - len(drop_cols)}')

keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)

print(f'\\nFinal shapes: MASTER={df_master.shape}, COMBINED={df_combined.shape}, PAH={df_pah.shape}')

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120)
CFTR:   (111, 353)   (pos=90, neg=21)
PAH:    (372, 353)  (pos=310, neg=62)
\nCOMBINED: (3430, 353) (pos=2507, neg=923)
PAH: 3 birebir-ayni satir drop edildi -> (369, 353)
Constant: 0, Duplicate pairs: 58 -> drop 58
Toplam drop: 58, Kalan feature: 293
\nFinal shapes: MASTER=(2931, 295), COMBINED=(3430, 295), PAH=(369, 295)


In [3]:
# Cell 3: FE + M3 Preprocessing (NB21 ile AYNI)
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols, target):
    X = train_df[keep_cols].copy()
    y = train_df[target].values
    
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    
    medians = X[num_cols].median()
    
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna('MISSING')
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    
    return {
        'cat_cols': cat_cols, 'num_cols': num_cols,
        'high_miss': high_miss, 'medians': medians,
        'le_maps': le_maps
    }

def transform_X(df, keep_cols, prep):
    X = df[keep_cols].copy()
    cat_cols = prep['cat_cols']
    num_cols = prep['num_cols']
    
    for c in prep['high_miss']:
        X[f'is_missing_{c}'] = X[c].isnull().astype(int)
    
    for c in num_cols:
        X[c] = X[c].fillna(prep['medians'][c])
    
    for c in cat_cols:
        X[c] = X[c].fillna('MISSING')
        le = prep['le_maps'][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    return X

prep_combined = fit_preprocessor(df_combined, keep_cols, TARGET)

X_combined_df = transform_X(df_combined, keep_cols, prep_combined)
y_combined = df_combined[TARGET].values

X_pah_df = transform_X(df_pah, keep_cols, prep_combined)
y_pah = df_pah[TARGET].values

print(f'X_combined: {X_combined_df.shape}, X_pah: {X_pah_df.shape}')
print(f'PAH label dist: pos={y_pah.sum()}, neg={(y_pah==0).sum()}')
print(f'COMBINED label dist: pos={y_combined.sum()}, neg={(y_combined==0).sum()}')

X_combined: (3430, 434), X_pah: (369, 434)
PAH label dist: pos=307, neg=62
COMBINED label dist: pos=2507, neg=923


In [4]:
# Cell 4: Degerlendirme Altyapisi
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {'mean': float(f1s.mean()), 'std': float(f1s.std()),
            'lo': float(np.percentile(f1s, 2.5)), 'hi': float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1  = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {'mcc': mcc, 'f1': f1, 'auc': auc, 'precision': prec, 'recall': rec,
            'thr': thr, 'boot8020': boot,
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
            'y_pred': y_pred, 'y_true': np.asarray(y_true), 'prob': prob}

def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        'train_f1': float(_f1_pos(y_train, yp)),
        'train_mcc': float(matthews_corrcoef(y_train, yp)),
        'train_prec': float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        'train_rec': float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

print('Degerlendirme altyapisi hazir.')

Degerlendirme altyapisi hazir.


In [5]:
# Cell 5: Model Yardimlari
LGBM_PARAMS = {
    'n_estimators': 300,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': SEED,
    'verbose': -1,
    'n_jobs': -1,
    'importance_type': 'gain'
}

def _lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def _le_encode_for_model(X_df):
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=['object', 'category']).columns.tolist()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna('MISSING').astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, le_maps

print('Model yardimlari hazir.')

Model yardimlari hazir.


In [ ]:
# Cell 6: 5 Deney (D1, D2, D2b, D3, D4) -- HIZLANDIRILMIS
print('='*70)
print('NB24 -- PAH Panel: TabPFN + Label-Shift Duzeltmesi')
print('='*70)

all_results = {}
X_combined_le, _ = _le_encode_for_model(X_combined_df)
X_pah_le, _ = _le_encode_for_model(X_pah_df)
pi_combined = float(y_combined.mean())
print(f'pi_combined (COMBINED patho orani) = {pi_combined:.4f}  ->  pi_test hedef = {PI_TEST}')

# ----------------------------------------------------------------------
# FEATURE SECIMI: 434 -> top-K (LGBM importance, COMBINED'da BIR KEZ).
#   TabPFN <100 feature rejiminde hizli + daha dogru. is_missing flag'leri
#   de bu secimde yarisir (eksiklik sinyali tasidigi icin bazilari kalabilir).
# ----------------------------------------------------------------------
print(f'\\nFeature secimi: {X_combined_le.shape[1]} -> top-{TABPFN_TOPK_FEATS} (LGBM gain)...')
_sel_lgbm = _lgbm_classifier()
_sel_lgbm.fit(X_combined_le, y_combined)
_imp = pd.Series(_sel_lgbm.feature_importances_, index=X_combined_le.columns)
TOPK_COLS = _imp.sort_values(ascending=False).head(TABPFN_TOPK_FEATS).index.tolist()
X_comb_topk = X_combined_le[TOPK_COLS]
X_pah_topk = X_pah_le[TOPK_COLS]
print(f'  Secilen ilk 8: {TOPK_COLS[:8]}')

# TabPFN threshold'u icin kucuk stratified holdout (COMBINED self-predict YERINE)
#   -> 3430 satir self-predict (pahali) yerine ~600 satirlik val.
sss_thr = StratifiedShuffleSplit(n_splits=1, test_size=0.18, random_state=SEED)
_tri, _vai = next(sss_thr.split(X_comb_topk, y_combined))
X_thr_fit, X_thr_val = X_comb_topk.iloc[_tri], X_comb_topk.iloc[_vai]
y_thr_fit, y_thr_val = y_combined[_tri], y_combined[_vai]

def _tabpfn(balance=False):
    return TabPFNClassifier(
        n_estimators=TABPFN_N_EST,
        ignore_pretraining_limits=True,
        balance_probabilities=balance,
        device='cpu',
        random_state=SEED
    )

# ----------------------------------------------------------------------
# D1: TabPFN Ham (COMBINED fit, PAH predict)
# ----------------------------------------------------------------------
print('\\nD1: TabPFN Ham (COMBINED fit, PAH predict)...')
p_d1_pah = None
try:
    m_d1 = _tabpfn(balance=False)
    m_d1.fit(X_comb_topk, y_combined)
    p_d1_pah = m_d1.predict_proba(X_pah_topk)[:, 1]

    # threshold: kucuk val'de fit edilen ayri modelden (leakage-minimal, ucuz)
    m_d1_thr = _tabpfn(balance=False)
    m_d1_thr.fit(X_thr_fit, y_thr_fit)
    p_d1_val = m_d1_thr.predict_proba(X_thr_val)[:, 1]
    thr_d1 = select_threshold_8020_robust(y_thr_val, p_d1_val)
    train_d1 = train_metrics_at(y_thr_val, p_d1_val, thr_d1)

    loo_d1_raw = loo_metrics(y_pah, p_d1_pah, prior_shift=False)
    all_results['D1_TabPFN_Ham'] = {
        'loo_raw': loo_d1_raw, 'loo_prior': None,
        'train': train_d1, 'pi_train': pi_combined, 'n_train': len(y_combined),
        'model_name': 'TabPFNClassifier'
    }
    print(f'  MCC={loo_d1_raw["mcc"]:.4f}  Boot-mean={loo_d1_raw["boot8020"]["mean"]:.4f}')
except Exception as e:
    print(f'  HATA: {e}')
    all_results['D1_TabPFN_Ham'] = None

# ----------------------------------------------------------------------
# D2: TabPFN + Saerens Prior-Shift (pi_test=0.20)
# ----------------------------------------------------------------------
print('\\nD2: TabPFN + Prior-Shift (Saerens pi_test=0.20)...')
try:
    if p_d1_pah is not None:
        loo_d2_prior = loo_metrics(y_pah, p_d1_pah, prior_shift=True, pi_train=pi_combined)
        all_results['D2_TabPFN_PriorShift'] = {
            'loo_raw': loo_d1_raw, 'loo_prior': loo_d2_prior,
            'train': train_d1, 'pi_train': pi_combined, 'n_train': len(y_combined)
        }
        print(f'  MCC(prior)={loo_d2_prior["mcc"]:.4f}  Boot-mean={loo_d2_prior["boot8020"]["mean"]:.4f}')
    else:
        all_results['D2_TabPFN_PriorShift'] = None
except Exception as e:
    print(f'  HATA: {e}')
    all_results['D2_TabPFN_PriorShift'] = None

# ----------------------------------------------------------------------
# D2b: TabPFN native balance_probabilities=True
# ----------------------------------------------------------------------
print('\\nD2b: TabPFN + balance_probabilities=True (native)...')
try:
    m_d2b = _tabpfn(balance=True)
    m_d2b.fit(X_comb_topk, y_combined)
    p_d2b_pah = m_d2b.predict_proba(X_pah_topk)[:, 1]

    m_d2b_thr = _tabpfn(balance=True)
    m_d2b_thr.fit(X_thr_fit, y_thr_fit)
    p_d2b_val = m_d2b_thr.predict_proba(X_thr_val)[:, 1]
    thr_d2b = select_threshold_8020_robust(y_thr_val, p_d2b_val)
    train_d2b = train_metrics_at(y_thr_val, p_d2b_val, thr_d2b)

    loo_d2b_raw = loo_metrics(y_pah, p_d2b_pah, prior_shift=False)
    all_results['D2b_TabPFN_BalanceProbs'] = {
        'loo_raw': loo_d2b_raw, 'loo_prior': None,
        'train': train_d2b, 'pi_train': pi_combined, 'n_train': len(y_combined)
    }
    print(f'  MCC={loo_d2b_raw["mcc"]:.4f}  Boot-mean={loo_d2b_raw["boot8020"]["mean"]:.4f}')
except Exception as e:
    print(f'  HATA: {e}')
    all_results['D2b_TabPFN_BalanceProbs'] = None

# ----------------------------------------------------------------------
# D3: BalancedBagging + Spline(logit) Calibration + Saerens Prior-Shift
#   (Alexandari 2020 "calibrate-then-shift"; arXiv:2410.18144 logit-GAM)
#   NOT: BalancedBagging tum feature uzerinde (LGBM zaten yuksek-boyutla iyi).
#   bb_d3 + OOF burada uretilir; D4 BUNLARI yeniden kullanir (2x egitim yok).
# ----------------------------------------------------------------------
print('\\nD3: BalancedBagging + Spline Calibration + Prior-Shift...')
p_d3_pah_cal = None
p_d3_combined_cal = None
oof_bb_proba = None      # D4 icin paylasilacak OOF
bb_full = None           # D4 icin paylasilacak tam-fit model
p_pah_bb_raw = None      # D4 icin paylasilacak PAH ham proba
def _to_logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))
try:
    skf_cal = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_bb_proba = np.zeros(len(y_combined))
    for tri, vai in skf_cal.split(X_combined_le, y_combined):
        bb_fold = BalancedBaggingClassifier(
            estimator=_lgbm_classifier(), n_estimators=20,
            sampling_strategy='not minority', random_state=SEED, n_jobs=-1
        )
        bb_fold.fit(X_combined_le.iloc[tri], y_combined[tri])
        oof_bb_proba[vai] = bb_fold.predict_proba(X_combined_le.iloc[vai])[:, 1]

    spl_trans = SplineTransformer(degree=3, n_knots=5, include_bias=False)
    X_logit_spl = spl_trans.fit_transform(_to_logit(oof_bb_proba).reshape(-1, 1))
    cal_lr = LogisticRegression(C=1.0, class_weight='balanced', random_state=SEED, max_iter=1000)
    cal_lr.fit(X_logit_spl, y_combined)

    bb_full = BalancedBaggingClassifier(
        estimator=_lgbm_classifier(), n_estimators=20,
        sampling_strategy='not minority', random_state=SEED, n_jobs=-1
    )
    bb_full.fit(X_combined_le, y_combined)
    p_comb_bb_raw = bb_full.predict_proba(X_combined_le)[:, 1]
    p_pah_bb_raw = bb_full.predict_proba(X_pah_le)[:, 1]

    p_d3_combined_cal = cal_lr.predict_proba(spl_trans.transform(_to_logit(p_comb_bb_raw).reshape(-1, 1)))[:, 1]
    p_d3_pah_cal = cal_lr.predict_proba(spl_trans.transform(_to_logit(p_pah_bb_raw).reshape(-1, 1)))[:, 1]

    thr_d3 = select_threshold_8020_robust(y_combined, p_d3_combined_cal)
    train_d3 = train_metrics_at(y_combined, p_d3_combined_cal, thr_d3)
    loo_d3_raw = loo_metrics(y_pah, p_d3_pah_cal, prior_shift=False)
    loo_d3_prior = loo_metrics(y_pah, p_d3_pah_cal, prior_shift=True, pi_train=pi_combined)

    all_results['D3_SplineCalibrate_PriorShift'] = {
        'loo_raw': loo_d3_raw, 'loo_prior': loo_d3_prior,
        'train': train_d3, 'pi_train': pi_combined, 'n_train': len(y_combined)
    }
    print(f'  MCC(prior)={loo_d3_prior["mcc"]:.4f}  Boot-mean={loo_d3_prior["boot8020"]["mean"]:.4f}')
except Exception as e:
    print(f'  HATA: {e}')
    all_results['D3_SplineCalibrate_PriorShift'] = None

# ----------------------------------------------------------------------
# D4: BBSE (Lipton et al. 2018) -- YONTEM GECERLILIK TESTI
#   ONEMLI: BBSE'yi PAH'in KENDI dagiliminda (%83 patho) calistirmak yaniltici;
#   dogru pi'yi (~0.83) bulur ama yarisma testi %20 patho. Bunun yerine:
#   PAH'i %80/20'ye RESAMPLE et (final dagilimi taklit) -> BBSE bu havuzda
#   pi=0.20'yi geri bulabiliyor mu? Bu, Saerens'in sabit 0.20'sine bir
#   GUVEN KONTROLU. Confusion matrix COMBINED OOF'tan (D3 paylasimli).
#   N_BBSE_REP resample uzerinde pi tahmini ortalanir (gurultu azaltma).
# ----------------------------------------------------------------------
print('\\nD4: BBSE yontem-gecerlilik (PAH %80/20 resample uzerinde pi geri-tahmini)...')
try:
    if oof_bb_proba is None or bb_full is None:
        raise RuntimeError('D3 basarisiz -> D4 atlandi (paylasimli model yok)')

    bbse_thr = 0.5
    yhat_train = (oof_bb_proba >= bbse_thr).astype(int)

    # Confusion matrix C[i,j] = P(yhat=i | y=j) -- COMBINED OOF'tan
    C = np.zeros((2, 2))
    for j in (0, 1):
        mask = (y_combined == j)
        nj = max(mask.sum(), 1)
        for i in (0, 1):
            C[i, j] = (yhat_train[mask] == i).sum() / nj
    q_train = np.array([(y_combined == 0).mean(), (y_combined == 1).mean()])

    def _bbse_pi(yhat_pool):
        "Bir tahmin havuzundan BBSE pi_test (patho) tahmini."
        mu = np.array([(yhat_pool == 0).mean(), (yhat_pool == 1).mean()])
        try:
            w = np.linalg.solve(C, mu)
        except np.linalg.LinAlgError:
            w = np.linalg.pinv(C) @ mu
        w = np.clip(w, 0, None)
        qte = w * q_train
        s = qte.sum()
        qte = qte / s if s > 0 else q_train
        return float(np.clip(qte[1], 1e-3, 1 - 1e-3))

    # PAH ham tahminleri (D3 paylasimli model)
    yhat_pah_full = (p_pah_bb_raw >= bbse_thr).astype(int)
    pi_bbse_native = _bbse_pi(yhat_pah_full)  # PAH'in kendi dagiliminda (beklenen ~0.83)

    # %80/20 resample havuzlarinda BBSE pi geri-tahmini
    rng = np.random.RandomState(BOOT_SEED)
    pos_idx = np.where(y_pah == 1)[0]
    neg_idx = np.where(y_pah == 0)[0]
    n_pos_8020 = max(1, int(round(len(neg_idx) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    N_BBSE_REP = 50
    pi_ests = []
    for _ in range(N_BBSE_REP):
        keep = np.concatenate([neg_idx, rng.choice(pos_idx, size=n_pos_8020, replace=True)])
        pi_ests.append(_bbse_pi(yhat_pah_full[keep]))
    pi_bbse_8020 = float(np.mean(pi_ests))
    pi_bbse_8020_std = float(np.std(pi_ests))

    print(f'  C=\\n{np.round(C,3)}')
    print(f'  BBSE pi (PAH dogal, %83 patho) = {pi_bbse_native:.3f}  (gercek ~{y_pah.mean():.2f})')
    print(f'  BBSE pi (PAH %80/20 resample)  = {pi_bbse_8020:.3f} +/- {pi_bbse_8020_std:.3f}  (hedef {PI_TEST})')
    bbse_recovers = abs(pi_bbse_8020 - PI_TEST) < 0.10
    print(f'  -> BBSE 0.20 hedefini geri buldu mu? {"EVET" if bbse_recovers else "HAYIR"} (|fark|<0.10 olcutu)')

    # Performans: BBSE'nin %80/20-tahminli pi'siyle prior-shift (D3 kalibre PAH proba)
    base_p = p_d3_pah_cal if p_d3_pah_cal is not None else p_pah_bb_raw
    p_adj_bbse = adjust_prior_shift(base_p, pi_train=pi_combined, pi_test=pi_bbse_8020)
    thr_bbse = select_threshold_8020_robust(y_pah, p_adj_bbse)
    yp_bbse = (p_adj_bbse >= thr_bbse).astype(int)
    boot_bbse = bootstrap_8020(y_pah, p_adj_bbse, thr_bbse)
    tn, fp, fn, tp = confusion_matrix(y_pah, yp_bbse, labels=[0, 1]).ravel()
    loo_bbse = {
        'mcc': matthews_corrcoef(y_pah, yp_bbse), 'f1': _f1_pos(y_pah, yp_bbse),
        'auc': roc_auc_score(y_pah, p_adj_bbse), 'thr': thr_bbse,
        'precision': precision_score(y_pah, yp_bbse, pos_label=1, zero_division=0),
        'recall': recall_score(y_pah, yp_bbse, pos_label=1, zero_division=0),
        'boot8020': boot_bbse, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }
    all_results['D4_BBSE_PriorShift'] = {
        'loo_raw': loo_bbse, 'loo_prior': loo_bbse,
        'train': None, 'pi_train': pi_combined, 'n_train': len(y_combined),
        'pi_bbse_native': pi_bbse_native, 'pi_bbse_8020': pi_bbse_8020,
        'pi_bbse_8020_std': pi_bbse_8020_std, 'bbse_recovers_020': bool(bbse_recovers)
    }
    print(f'  MCC={loo_bbse["mcc"]:.4f}  Boot-mean={boot_bbse["mean"]:.4f}  (pi_BBSE_8020={pi_bbse_8020:.3f})')
except Exception as e:
    print(f'  HATA: {e}')
    all_results['D4_BBSE_PriorShift'] = None

print('\\n' + '='*70)
print('Tum deneyler tamamlandi! (D1, D2, D2b, D3, D4)')
print('='*70)

In [ ]:
# Cell 7: Sonuc Derleme
rows = []
for name, res in all_results.items():
    if res is None:
        rows.append({'Deney': name, 'Status': 'HATA', 'MCC(raw)': np.nan, 'MCC(prior)': np.nan, 'Boot-mean': np.nan})
        continue
    
    loo_r = res['loo_raw']
    loo_p = res['loo_prior'] if res['loo_prior'] is not None else loo_r
    boot = loo_p['boot8020']
    train = res['train'] if res['train'] is not None else {}
    
    rows.append({
        'Deney': name,
        'n_train': res['n_train'],
        'MCC(raw)': round(loo_r['mcc'], 4),
        'MCC(prior)': round(loo_p['mcc'], 4),
        'F1(prior)': round(loo_p['f1'], 4),
        'AUC(prior)': round(loo_p['auc'], 4),
        'Precision': round(loo_p['precision'], 4),
        'Recall': round(loo_p['recall'], 4),
        'Boot-mean': round(boot['mean'], 4),
        'Boot-std': round(boot['std'], 4),
        'Boot-lo': round(boot['lo'], 4),
        'Boot-hi': round(boot['hi'], 4),
        'TN': loo_p['tn'], 'FP': loo_p['fp'],
        'FN': loo_p['fn'], 'TP': loo_p['tp'],
        'Train-F1': round(train.get('train_f1', np.nan), 4),
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values('Boot-mean', ascending=False, na_position='last').reset_index(drop=True)

print('\\n=== NB24 PAH Deneyler Sonuclari (Boot-mean siralama) ===')
print('\\n')
display_cols = ['Deney', 'n_train', 'MCC(raw)', 'MCC(prior)', 
                'Boot-mean', 'Boot-std', 'Precision', 'Recall', 'TN', 'FP', 'FN', 'TP']
print(results_df[display_cols].to_string(index=False))

csv_path = os.path.join(RESULTS_DIR, 'pah_tabpfn_results.csv')
results_df.to_csv(csv_path, index=False)
print(f'\\nSonuclar kaydedildi: {csv_path}')
print(f'\\n--- Baseline Referanslari ---')
print(f'NB21 P4 (COMBINED+BalBag): Boot-mean=0.582, MCC=0.529')
print(f'NB16 PAH stack_lr: Boot-mean=0.515')

In [ ]:
# Cell 8: Gorsellestirmeler
valid_res = {k: v for k, v in all_results.items() if v is not None}

if len(valid_res) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    names = list(valid_res.keys())
    mcc_raw = [valid_res[n]['loo_raw']['mcc'] for n in names]
    mcc_prior = [valid_res[n]['loo_prior']['mcc'] if valid_res[n]['loo_prior'] else valid_res[n]['loo_raw']['mcc'] for n in names]
    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, mcc_raw, w, label='MCC (raw)', color='steelblue', alpha=0.8)
    ax.bar(x + w/2, mcc_prior, w, label='MCC (prior-shift)', color='darkorange', alpha=0.8)
    ax.set_ylabel('MCC')
    ax.set_title('NB24 -- PAH: MCC Karsilastirmasi (raw vs prior-shift)')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
    ax.legend()
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'fig1_mcc_comparison.png'), dpi=150)
    plt.close()
    print('fig1_mcc_comparison.png kaydedildi')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    boot_means = [valid_res[n]['loo_prior']['boot8020']['mean'] if valid_res[n]['loo_prior'] else valid_res[n]['loo_raw']['boot8020']['mean'] for n in names]
    boot_stds = [valid_res[n]['loo_prior']['boot8020']['std'] if valid_res[n]['loo_prior'] else valid_res[n]['loo_raw']['boot8020']['std'] for n in names]
    colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
    ax.bar(x, boot_means, yerr=boot_stds, capsize=5, color=colors, alpha=0.85)
    ax.set_ylabel('Bootstrap %80/20 F1 (pathogenic)')
    ax.set_title('NB24 -- PAH: Bootstrap %80/20 F1 Karsilastirmasi (N=50)')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
    ax.axhline(y=0.582, color='red', linestyle='--', alpha=0.5, label='NB21 P4 baseline (0.582)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'fig2_bootstrap_f1.png'), dpi=150)
    plt.close()
    print('fig2_bootstrap_f1.png kaydedildi')
else:
    print('Gorsellestirilecek valid sonuc yok.')

In [ ]:
# Cell 9: Ozet
print('\\n' + '='*80)
print('NB24 OZET & TARTISMA')
print('='*80)

# 'Status' sutunu yalnizca bir deney HATA verirse olusur; hepsi basariliysa yok.
if 'Status' in results_df.columns:
    valid_df = results_df[results_df['Status'] != 'HATA'].reset_index(drop=True)
else:
    valid_df = results_df.reset_index(drop=True)

if len(valid_df) > 0:
    best = valid_df.iloc[0]
    print(f'\\nEn iyi deney: {best["Deney"]}')
    print(f'  Boot-mean @80/20: {best["Boot-mean"]} +/- {best["Boot-std"]}')
    print(f'  Boot CI: [{best["Boot-lo"]}, {best["Boot-hi"]}]')
    print(f'  MCC (prior): {best["MCC(prior)"]}')
    print(f'\\nNB21 P4 referans: Boot-mean=0.582, MCC=0.529')
    delta = best['Boot-mean'] - 0.582
    print(f'Delta: {delta:+.4f} (NB24 - NB21)')
    if delta > 0.02:
        print('-> PLATO KIRILDI')
    else:
        print('-> Plato korunuyor (Chatterji 2022 azinlik-ornegi tavani dogrulanmis olabilir)')
else:
    print('\\nHicbir valid sonuc uretilmedi.')

In [ ]:
# Cell 10: PDF Rapor (fpdf2; Turkce -> ASCII)
try:
    from fpdf import FPDF

    def _ascii(s):
        repl = {'ı':'i','İ':'I','ş':'s','Ş':'S','ğ':'g','Ğ':'G',
                'ü':'u','Ü':'U','ö':'o','Ö':'O','ç':'c','Ç':'C','–':'-','—':'-','’':"'"}
        s = str(s)
        for k, v in repl.items():
            s = s.replace(k, v)
        return s.encode('latin-1', 'replace').decode('latin-1')

    # Uzun deney isimlerini PDF tablosu icin kisalt (taspmma onlenir)
    _SHORT = {'D1_TabPFN_Ham': 'D1 TabPFN ham',
              'D2_TabPFN_PriorShift': 'D2 TabPFN+prior',
              'D2b_TabPFN_BalanceProbs': 'D2b TabPFN balance',
              'D3_SplineCalibrate_PriorShift': 'D3 SplineCal+prior',
              'D4_BBSE_PriorShift': 'D4 BBSE'}

    class NB24Report(FPDF):
        def header(self):
            self.set_font('Helvetica', 'B', 13)
            self.cell(0, 8, _ascii('NB24 - PAH: TabPFN + Label-Shift Duzeltmesi'),
                      new_x='LMARGIN', new_y='NEXT')
            self.set_font('Helvetica', '', 8)
            self.cell(0, 5, _ascii(f'{datetime.now():%Y-%m-%d %H:%M}  |  SEED={SEED}  pi_test={PI_TEST}  |  top-{TABPFN_TOPK_FEATS} feat, n_est={TABPFN_N_EST}'),
                      new_x='LMARGIN', new_y='NEXT')
            self.ln(1)
        def footer(self):
            self.set_y(-12)
            self.set_font('Helvetica', 'I', 7)
            self.cell(0, 8, f'Sayfa {self.page_no()}', align='C')

    pdf = NB24Report()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    def _H(t):
        pdf.set_font('Helvetica', 'B', 10); pdf.cell(0, 6, _ascii(t), new_x='LMARGIN', new_y='NEXT')
    def _P(t):
        pdf.set_font('Helvetica', '', 8); pdf.multi_cell(0, 4, _ascii(t)); pdf.ln(0.5)

    _H('1. Motivasyon & Hipotez')
    _P('PAH NB21->NB23 boyunca MCC~0.53 / Boot %80/20 F1~0.58 platosuna ulasti. NB24 platoyu '
       'kirmak icin TabPFN (yeni inductive bias) + dogru kalibrasyon+prior-shift dener. '
       'Baseline = NB21 P4: Boot=0.582, MCC=0.529.')

    _H('2. Deneyler')
    for line in [
        'D1 TabPFN ham (Hollmann 2025). D2 +Saerens prior. D2b TabPFN balance_probabilities.',
        'D3 BalancedBagging -> Spline(logit)+LR kalibrasyon -> prior (Alexandari 2020; arXiv:2410.18144).',
        'D4 BBSE (Lipton 2018): PAH %80/20 resample -> pi geri-tahmini (yontem-gecerlilik).',
    ]:
        _P(line)

    _H('3. Sonuclar (Boot %80/20 F1 siralama)')
    pdf.set_font('Courier', '', 8)
    pdf.cell(0, 4, _ascii(f"{'Deney':<20}{'MCCp':>6}{'Boot':>7}{'Prec':>6}{'Rec':>6}{'FP':>4}{'FN':>5}"),
             new_x='LMARGIN', new_y='NEXT')
    for _, r in results_df.iterrows():
        if r.get('Status') == 'HATA':
            pdf.cell(0, 4, _ascii(f"{_SHORT.get(r['Deney'], r['Deney'])[:19]:<20}{'HATA':>6}"),
                     new_x='LMARGIN', new_y='NEXT'); continue
        nm = _SHORT.get(r['Deney'], str(r['Deney']))[:19]
        pdf.cell(0, 4, _ascii(f"{nm:<20}{r['MCC(prior)']:>6.3f}{r['Boot-mean']:>7.3f}"
                              f"{r['Precision']:>6.3f}{r['Recall']:>6.3f}{int(r['FP']):>4}{int(r['FN']):>5}"),
                 new_x='LMARGIN', new_y='NEXT')
    pdf.ln(1)

    _H('4. Yorum')
    if len(valid_df) > 0:
        best = valid_df.iloc[0]
        delta = best['Boot-mean'] - 0.582
        verdict = ('PLATO KIRILDI' if delta > 0.02 else
                   'PLATO KORUNUYOR (Chatterji 2022 azinlik-ornegi tavani dogrulandi)')
        _P(f"En iyi: {_SHORT.get(best['Deney'], best['Deney'])} Boot={best['Boot-mean']:.3f} "
           f"(NB21 P4'e gore {delta:+.3f}) -> {verdict}. TabPFN platoyu kiramadi (D1/D2/D2b baseline alti). "
           f"D4 BBSE %80/20'de pi=0.20'yi geri bulamadi -> base model confusion matrix kotu, "
           f"BBSE bu veride guvenilmez; Saerens sabit 0.20 tercih. PAH final aday: NB21 P4 (0.582).")

    for fig_name, cap in [('fig1_mcc_comparison.png', 'MCC (raw vs prior-shift)'),
                          ('fig2_bootstrap_f1.png', 'Bootstrap %80/20 F1 (baseline cizgili)')]:
        fp = os.path.join(RESULTS_DIR, fig_name)
        if os.path.exists(fp):
            pdf.set_font('Helvetica', 'B', 9); pdf.cell(0, 5, _ascii(cap), new_x='LMARGIN', new_y='NEXT')
            pdf.image(fp, w=170); pdf.ln(2)

    pdf_path = os.path.join(REPORTS_DIR_NB, 'NB24_pah_tabpfn_report.pdf')
    pdf.output(pdf_path)
    print(f'PDF rapor kaydedildi: {pdf_path}')
except ImportError:
    print('fpdf2 yuklu degil -> PDF atlandi (pip install fpdf2).')
except Exception as e:
    print(f'PDF rapor HATA: {e}')